# flowbars vs mlfinpy — bar construction comparison

Compares [flowbars](https://github.com/syedqaisarjalil/flowbars) against
[mlfinpy](https://pypi.org/project/mlfinpy/) — the free, open-source
reimplementation of the paid MlFinLab — on identical synthetic tick data.

**Findings** (each reproducible below):

1. mlfinpy **drops threshold-crossing overflow**; flowbars carries it (AFML-correct).
2. Standard bars therefore **diverge** exactly where overflow matters.
3. Information-driven bars use **different formulas** — flowbars: two-component
   EWMA `T = E[T] × |E[θ]|` (AFML §2.4); mlfinpy: a single EWMA over raw
   per-tick signed imbalance.
4. flowbars' numba backend is **~6.6× faster** than mlfinpy's pure-Python loop.


## Prerequisites

`mlfinpy` requires Python ≥ 3.11. Run this notebook in a 3.11+ environment:

```bash
conda create -n flowbars311 python=3.11 -y
conda activate flowbars311
pip install mlfinpy scikit-learn
pip install /path/to/flowbars          # non-editable install
```

Headless re-run:

```bash
jupyter nbconvert --to notebook --execute notebooks/01_compare_mlfinpy.ipynb
```


In [ ]:
from __future__ import annotations

import time

import numpy as np
import pandas as pd

import flowbars
from flowbars import (
    SchemaMapping,
    compute_dollar_bars,
    compute_imbalance_dollar_bars,
    compute_imbalance_volume_bars,
    compute_run_dollar_bars,
    compute_run_volume_bars,
    compute_tick_bars,
    compute_volume_bars,
)
from flowbars.bars.accumulators import VolumeAccumulator
from flowbars.bars.constructor import BaseBarConstructor
from flowbars.thresholds import StaticThresholdEstimator

import mlfinpy.data_structure as ds

print("flowbars", flowbars.__version__)


In [ ]:
# Fixed seed — identical data for both libraries
N = 200_000
rng = np.random.default_rng(42)
ts = np.cumsum(rng.integers(50, 200, N))  # int ms, strictly increasing
price = 4000.0 + np.cumsum(rng.normal(0.0, 0.1, N))
price = np.clip(price, 1.0, None)
volume = rng.integers(1, 10, N).astype(float)

# mlfinpy wants exactly 3 columns in order: date_time, price, volume
df_ml = pd.DataFrame({"date_time": ts, "price": price, "volume": volume})

# flowbars wants an explicit schema; timestamps are int ms
df_fb = pd.DataFrame({"ts": ts, "price": price, "volume": volume})
SCHEMA = SchemaMapping({"timestamp": "ts", "price": "price", "volume": "volume"})

TOTAL_VOLUME = float(volume.sum())
TOTAL_DOLLAR = float((price * volume).sum())

print(f"{N:,} ticks | total volume {TOTAL_VOLUME:,.0f} | total dollar {TOTAL_DOLLAR:,.0f}")


def _len(x):
    """mlfinpy info-driven functions return a (df, thresholds) tuple."""
    return len(x[0]) if isinstance(x, tuple) else len(x)


## Finding 1 — mlfinpy drops overflow

Hand-verifiable case: volumes `[7, 7, 7]`, threshold `10`.
flowbars carries the excess (`7+7=14`, excess `4` rolls into the next bar);
mlfinpy resets to zero (excess `4` discarded, so the last `7` never reaches `10`).


In [ ]:
small_ml = pd.DataFrame({"date_time": [0, 1, 2], "price": [100.0, 101.0, 102.0], "volume": [7.0, 7.0, 7.0]})
small_fb = pd.DataFrame({"ts": [0, 1, 2], "price": [100.0, 101.0, 102.0], "volume": [7.0, 7.0, 7.0]})

fb = compute_volume_bars(small_fb, threshold=10, schema=SCHEMA, watermark=None)
ml = ds.get_volume_bars(small_ml, threshold=10, verbose=False)

pd.DataFrame(
    {
        "library": ["flowbars", "mlfinpy"],
        "num_bars": [len(fb), len(ml)],
        "bar0_volume": [fb.iloc[0]["volume"], ml.iloc[0]["volume"]],
        "bar0_ticks": [fb.iloc[0]["num_ticks"], ml.iloc[0]["cum_ticks"]],
        "has_2nd_bar": [len(fb) >= 2, len(ml) >= 2],
    }
)


## Finding 2 — standard bars diverge where overflow matters

Thresholds are chosen for ~20 bars. The last (partial) bar is dropped by both
libraries; the difference is the accumulated overflow that mlfinpy discards.


In [ ]:
standard = [
    ("tick", compute_tick_bars, ds.get_tick_bars, {"threshold": 10_000}),
    ("volume", compute_volume_bars, ds.get_volume_bars, {"threshold": TOTAL_VOLUME / 20.0}),
    ("dollar", compute_dollar_bars, ds.get_dollar_bars, {"threshold": TOTAL_DOLLAR / 20.0}),
]

rows = []
for name, fb_fn, ml_fn, kw in standard:
    thresh = kw["threshold"]
    fb = fb_fn(df_fb, threshold=thresh, schema=SCHEMA, watermark=None)
    ml = ml_fn(df_ml, threshold=thresh, verbose=False)
    rows.append(
        (
            name,
            len(fb),
            len(ml),
            thresh,
            float(fb["dollar_value"].sum()),
            float(ml["cum_dollar_value"].sum()),
        )
    )

pd.DataFrame(
    rows,
    columns=["bar_type", "flowbars_bars", "mlfinpy_bars", "threshold", "fb_dollar_sum", "ml_dollar_sum"],
)


## Finding 3 — information-driven bars: different formulas

flowbars: two-component EWMA `T = E[T] × |E[θ]|` (AFML §2.4), where `θ` is the
bar-level signed proportion.

mlfinpy: a single EWMA over the raw per-tick signed imbalance. On balanced
random-walk data that EWMA converges toward zero, collapsing the threshold to
~one bar per tick (see `imbalance_*` rows).


In [ ]:
info = [
    ("imbalance_volume", compute_imbalance_volume_bars, ds.get_ema_volume_imbalance_bars),
    ("imbalance_dollar", compute_imbalance_dollar_bars, ds.get_ema_dollar_imbalance_bars),
    ("run_volume", compute_run_volume_bars, ds.get_ema_volume_run_bars),
    ("run_dollar", compute_run_dollar_bars, ds.get_ema_dollar_run_bars),
]

rows = []
for name, fb_fn, ml_fn in info:
    fb = fb_fn(df_fb, span=20.0, warmup_bars=10, schema=SCHEMA, watermark=None)
    ml = ml_fn(df_ml, num_prev_bars=3, expected_imbalance_window=10000, exp_num_ticks_init=20000, verbose=False)
    rows.append((name, len(fb), _len(ml)))

pd.DataFrame(rows, columns=["bar_type", "flowbars_bars", "mlfinpy_bars"])


## Finding 4 — performance (volume bars, best of 3)

In [ ]:
V_THRESH = TOTAL_VOLUME / 20.0


def timeit(fn, *args, repeat=3, **kwargs):
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args, **kwargs)
        best = min(best, time.perf_counter() - t0)
    return best


# warm up both backends
compute_volume_bars(df_fb, threshold=V_THRESH, schema=SCHEMA, watermark=None)
acc = VolumeAccumulator(bar_type="volume")
est = StaticThresholdEstimator(threshold=V_THRESH)
ctor = BaseBarConstructor(acc, est, schema=SCHEMA, backend="numba", watermark=None)
ctor.batch(df_fb)  # compile numba

t_fb_py = timeit(compute_volume_bars, df_fb, threshold=V_THRESH, schema=SCHEMA, watermark=None)
t_fb_nb = timeit(ctor.batch, df_fb)
t_ml = timeit(ds.get_volume_bars, df_ml, threshold=V_THRESH, verbose=False)

pd.DataFrame(
    [
        ("flowbars (python)", t_fb_py * 1000),
        ("flowbars (numba)", t_fb_nb * 1000),
        ("mlfinpy", t_ml * 1000),
    ],
    columns=["library", "ms"],
)
